# Cheat-sheet Data Mining — Esame

Tutto quello che serve per le tracce d'esame, organizzato per argomento.
Ogni sezione: import + pattern d'uso + note sui punti dove si sbaglia facilmente.

## 1. Import completi (copia-incolla a inizio esame)

In [ ]:
# --- Dati e manipolazione ---
import pandas as pd
import numpy as np

# --- Grafici ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Dataset di scikit-learn ---
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_digits
from sklearn.datasets import fetch_california_housing   # attenzione: fetch_, non load_

# --- Model selection e preprocessing ---
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

# --- Clustering ---
from sklearn.cluster import KMeans

# --- Classificatori ---
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

# --- Metriche ---
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# --- Reti neurali (PyTorch) ---
import torch
from torch import nn
import torch.nn.functional as F

## 2. Caricare un dataset sklearn in un DataFrame

I dataset sklearn sono oggetti `Bunch` con 3 campi chiave: `.data` (matrice features), `.target` (label), `.feature_names` (nomi colonne).

In [ ]:
# Pattern standard (vale per iris, wine, breast_cancer, digits)
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

# California housing: si carica con fetch_ (il target e' un valore continuo, non una classe)
california = fetch_california_housing()
df_cal = pd.DataFrame(california.data, columns=california.feature_names)
df_cal['target'] = california.target

# digits: per i classificatori spesso non serve il DataFrame, bastano X e y
digits = load_digits()
X_dig, y_dig = digits.data, digits.target

## 3. Esplorazione del DataFrame

In [ ]:
print(df.head(10))      # prime 10 righe (default: 5)
print(df.tail(10))      # ultime 10 righe
df.info()               # dtypes, valori non-null, uso memoria (stampa da sola)
print(df.describe())    # count, mean, std, min, quartili, max
print(df.shape)         # (righe, colonne)
print(df.columns)       # nomi colonne
print(df.mean(numeric_only=True))   # media per colonna
print(df.corr(numeric_only=True))   # matrice di correlazione

## 4. Grafici

Schema fisso: disegna -> `xlabel`/`ylabel`/`title` -> `plt.show()`.

In [ ]:
# --- Istogramma di una colonna ---
df['mean radius'].hist()            # equivalente: plt.hist(df['mean radius'])
plt.xlabel('Mean Radius')
plt.ylabel('Frequency')
plt.title('Histogram of Mean Radius')
plt.show()

In [ ]:
# --- Bar plot dei valori medi di ogni feature ---
mean_values = df.mean(numeric_only=True)
mean_values.plot(kind='bar')
plt.xlabel('Features')
plt.ylabel('Mean Value')
plt.title('Mean Values of Features')
plt.show()

In [ ]:
# --- Scatter plot (tipico per visualizzare i cluster) ---
# c= colora i punti in base a una colonna (es. il cluster assegnato)
plt.scatter(df.iloc[:, 0], df.iloc[:, 1], c=df['target'], cmap='viridis')
plt.xlabel(df.columns[0])
plt.ylabel(df.columns[1])
plt.title('Scatter Plot')
plt.show()

In [ ]:
# --- Heatmap della correlazione ---
# NOTA: la cmap giusta si chiama 'coolwarm'. Nella soluzione della traccia
# T_2026_04_01 c'e' scritto 'warmcool': quel nome NON esiste e darebbe errore.
correlation_matrix = df.corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

## 5. K-Means (clustering)

Differenza chiave:
- `fit(X)` -> addestra soltanto
- `fit_predict(X)` -> addestra E restituisce il cluster di ogni record (comodo per assegnarlo a una colonna)
- `inertia_` -> somma delle distanze quadratiche dai centroidi (piu' bassa = cluster piu' compatti)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
df['Cluster'] = kmeans.fit_predict(df.drop('target', axis=1))

print(kmeans.inertia_)           # inerzia
print(kmeans.cluster_centers_)   # coordinate dei centroidi

# Visualizzazione: scatter colorato per cluster
plt.scatter(df.iloc[:, 0], df.iloc[:, 1], c=df['Cluster'], cmap='viridis')
plt.xlabel(df.columns[0])
plt.ylabel(df.columns[1])
plt.title('K-Means Clustering')
plt.show()

In [ ]:
# --- Elbow method: scegliere il numero di cluster ---
# Si prova k crescente e si cerca il "gomito" della curva dell'inerzia
X = df.drop(['target', 'Cluster'], axis=1)
inertias = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.plot(range(1, 11), inertias, marker='o')
plt.xlabel('Numero di cluster k')
plt.ylabel('Inerzia')
plt.title('Elbow Method')
plt.show()

## 6. Train/test split

`test_size=0.2` -> 20% test, 80% training. `random_state` fissa il risultato (riproducibile).

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 7. Classificatori

Tutti seguono lo stesso schema: **definisci -> `fit(X_train, y_train)` -> `predict(X_test)`**.
Cambiano solo il costruttore e i suoi iperparametri.

In [ ]:
# Decision Tree (iperparametro tipico: max_depth)
dt = DecisionTreeClassifier(max_depth=15)

# KNN (iperparametro tipico: n_neighbors)
knn = KNeighborsClassifier(n_neighbors=5)

# SVM (iperparametro tipico: kernel -> 'linear' o 'rbf')
svm = SVC(kernel='linear')

# Random Forest (iperparametro tipico: n_estimators = numero di alberi)
rf = RandomForestClassifier(n_estimators=80)

# Addestramento e predizione (uguale per tutti)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

In [ ]:
# --- Voting Classifier: combina piu' modelli ---
# voting='hard' -> vince la classe piu' votata
# voting='soft' -> media delle probabilita' (i modelli devono supportare predict_proba)
voting = VotingClassifier(estimators=[
    ('dt', dt),
    ('knn', knn)
], voting='hard')

voting.fit(X_train, y_train)
y_pred_voting = voting.predict(X_test)

## 8. Metriche di valutazione

ATTENZIONE a `f1_score`: il default e' `average='binary'`, funziona solo con 2 classi.
Con dataset multiclasse (wine, digits, iris) serve `average='macro'` (o `'weighted'`).

In [ ]:
# Accuratezza: frazione di predizioni corrette
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

# F1 score: media armonica di precision e recall
print(f1_score(y_test, y_pred))                    # solo 2 classi (es. breast_cancer)
# print(f1_score(y_test, y_pred, average='macro')) # multiclasse (wine, digits, iris)

In [ ]:
# --- Confusion matrix + heatmap ---
# annot=True scrive i numeri nelle celle, fmt='d' li formatta come interi
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 9. Cross-validation

Niente train/test split manuale: `cross_val_score` divide da sola in `cv` fold,
addestra `cv` volte e restituisce un array di score (uno per fold).

In [ ]:
model = RandomForestClassifier(n_estimators=80)

scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print(scores)                                       # uno score per fold
print(f'Cross-validated Accuracy: {scores.mean()}') # quasi sempre chiedono la media

## 10. Rete neurale con PyTorch

Schema fisso:
1. Classe che estende `nn.Module`
2. Nel `__init__`: chiama `super().__init__()` e definisci i layer `nn.Linear(in, out)` — l'output di un layer deve combaciare con l'input del successivo
3. Nel `forward`: applica i layer in sequenza con le attivazioni

Nota su `softmax`: `dim=1` normalizza lungo le colonne, cioe' ogni RIGA somma a 1 -> una distribuzione di probabilita' per ogni campione del batch. E' quello che vuoi nel 99% dei casi.

In [ ]:
class MyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(128, 64)   # input 128 -> hidden 64
        self.fc2 = nn.Linear(64, 16)    # hidden 64 -> hidden 16
        self.fc3 = nn.Linear(16, 4)     # hidden 16 -> output 4

    def forward(self, x):
        x = torch.relu(self.fc1(x))     # equivalente: F.relu(...)
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return torch.softmax(x, dim=1)  # dim=1: ogni riga somma a 1

# Verifica rapida: batch di 5 campioni con 128 feature
net = MyNetwork()
out = net(torch.rand(5, 128))
print(out.shape)        # torch.Size([5, 4])
print(out.sum(dim=1))   # tutti 1.0

## 11. Bonus: StandardScaler

KNN, SVM e K-Means si basano su distanze: feature con scale diverse dominano il calcolo.
Lo scaling le porta tutte a media 0 e deviazione standard 1.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Con train/test split: fit SOLO sul training (il test non deve influenzare lo scaler)
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)